# Combined Datasets

ENTSO-E, JAO and Investing.com are joined into the three feature sets for TFT and for trees, and into `epf` for Poland. `PL_price` is the Polish day-ahead price at hour t and is the target.

* **D1** — the Polish market and macro factors: ENTSO-E Poland, then Investing.com.
* **D2** — D1 plus Polish market coupling from JAO.
* **D3** — D2 plus the foreign ENTSO-E load forecasts and spreads, and the foreign JAO net positions and flows.

Each step takes the feature files saved for it, and for every earlier step. Columns follow the target, then the blocks in the order D1, D2, D3, each with ENTSO-E before JAO, and the calendar last.

**TFT.** 19 columns in D1, 30 in D2 and 46 in D3, with the calendar and `time_idx`. `time_idx` is the index column of the PyTorch dataset and is not a feature.

**Trees.** Built from the TFT frame of the same step, so both models see the same data. Trees get every TFT column and, in addition, columns that put the same data in a form a tree can split on (see `fe_trees.py`):

* D1: `PL_price_d1_mean`, `PL_price_d1_min`, `PL_price_d1_max`, `PL_price_mean7d_same_hour`, `PL_price_std7d`, `PL_resload_persist`, and the 7-day changes `ttf_chg7d`, `eua_chg7d`, `coal_chg7d`, `eurpln_chg7d`.
* D2 adds `headroom_import_PL`, `headroom_export_PL`, `gen_vload_balance_PL`, `np_PL_delta24`, `gen_PL_delta24` and `vload_PL_delta24`.
* D3 adds nothing beyond the TFT columns.

`epf` keeps `PL_load_fc`, `PL_load_fc_delta24`, `PL_wind_fc_lag24`, `PL_solar_fc_lag24` and `PL_gen_fc_lag24` from D1, and adds `PL_gen_fc_balance`, equal to `PL_gen_fc_lag24 - (PL_solar_fc_lag24 + PL_load_fc_delta24)`. It has no calendar columns and no `time_idx`.

The join is an inner join on the UTC index. The saved index is naive Europe/Warsaw time, one row for every hour from 00:00 to 23:00. On a spring-change day the missing 02:00 is the interpolation of the neighboring hours. On an autumn-change day the two 02:00 hours are averaged into one. `hour`, `dow`, `month`, `is_holiday` and `is_bridge` are then read from that clock. TFT and tree datasets are written to `data/final/{tft|trees}_{d1|d2|d3}_{split}.parquet`. `epf` is written as CSV with the same naive index. The download date range is not part of the file name.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd

from src.config import (
    DOWNLOAD_END_DATE,
    DOWNLOAD_START_DATE,
    FULL_DATASET_END_DATE,
    FULL_DATASET_START_DATE,
)
from src.utils.feature_engineering.fe_calendar import (
    add_calendar_features,
    add_time_idx,
)
from src.utils.file_io import (
    final_dataset_path,
    inspect_merged_file,
    load_final_dataset,
    save_merged_data,
)
from src.utils.merging import (
    DATASETS,
    cut_to_dates,
    merge_features,
    split_dataset,
    to_warsaw_hourly,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

## 1. Joining the Sources

Each dataset takes the feature files already saved for its step and the steps before it. `PL_price` comes from the preprocessed Polish day-ahead price and is placed first.

NaN counts below are the lag warmup at the start of the download window. The modeling window starts later, once those lags exist.

In [3]:
merged = {}
for model, scope in DATASETS:
    df = merge_features(model, scope, date_range)
    merged[(model, scope)] = df
    print(f"{model}({scope})  {df.shape}  NaN {int(df.isna().sum().sum())}")

tft(d1)  (38281, 14)  NaN 386
tft(d2)  (38281, 25)  NaN 395
tft(d3)  (38281, 41)  NaN 526
trees(d1)  (38281, 24)  NaN 1491
trees(d2)  (38281, 41)  NaN 1578
trees(d3)  (38281, 57)  NaN 1709
epf(pl)  (38281, 7)  NaN 123


## 2. UTC Join

The joined index is UTC and already hourly. It is kept only through the date cut. The saved frame uses the Polish clock.

In [4]:
reference = merged[("tft", "d1")]
steps = reference.index.to_series().diff().dropna()
same_index = all(df.index.equals(reference.index) for df in merged.values())

print(f"{reference.index.min()} -> {reference.index.max()}  {reference.index.tz}")
print(
    f"rows {len(reference)}  hourly {(steps == pd.Timedelta(hours=1)).all()}  unique {reference.index.is_unique}  same index {same_index}"
)

2022-06-08 22:00:00+00:00 -> 2026-10-20 22:00:00+00:00  UTC
rows 38281  hourly True  unique True  same index True


## 3. Modeling Window

`FULL_DATASET_START_DATE`–`FULL_DATASET_END_DATE` is the window the splits are cut from. Both dates are Europe/Warsaw calendar dates. The slice starts at local midnight and ends at local 23:00. `to_warsaw_hourly` then drops the timezone, averages the two autumn 02:00 hours, and interpolates the missing spring 02:00. Every saved day has 24 hours and the index has no timezone.

In [5]:
window = {
    key: to_warsaw_hourly(
        cut_to_dates(df, FULL_DATASET_START_DATE, FULL_DATASET_END_DATE)
    )
    for key, df in merged.items()
}

reference = window[("tft", "d1")]
steps = reference.index.to_series().diff().dropna()
day_lengths = reference.index.to_series().groupby(reference.index.date).size()
same_index = all(df.index.equals(reference.index) for df in window.values())
print(f"{reference.index.min()} -> {reference.index.max()}  tz {reference.index.tz}")
print(
    f"{reference.shape}  NaN {int(reference.isna().sum().sum())}  hourly {(steps == pd.Timedelta(hours=1)).all()}  same index {same_index}"
)
print("hours per day", int(day_lengths.min()), int(day_lengths.max()))
for day in ("2024-03-31", "2024-10-27"):
    hours = reference.loc[day].index.hour
    print(day, list(hours) == list(range(24)))

2022-06-20 00:00:00 -> 2026-10-18 23:00:00  tz None
(37968, 14)  NaN 0  hourly True  same index True
hours per day 24 24
2024-03-31 True
2024-10-27 True


## 4. Calendar Features

Read from the naive Warsaw index and attached to the TFT and tree datasets. Every day, including the clock-change days, has hours 0–23. `epf` keeps the same index and does not get these columns.

**TFT and trees.** `hour`, `dow`, `month`, `is_holiday` and `is_bridge`. `dow` and `month` are integers. `is_holiday` and `is_bridge` are 0/1. No sin/cos encoding and no one-hot columns. The spring 02:00 row is the interpolated hour. The autumn 02:00 row is the average of the two real hours.

`hour` stays in the tree frames so the rows can be split into 24 sets, one model per clock hour, if needed.

**`time_idx`.** TFT datasets only. It counts consecutive steps from 0 at the first hour of the full window and is the index of the PyTorch dataset, not a feature. Tree datasets do not have it.

**EPF.** `PL_price`, the five Polish forecast columns and `PL_gen_fc_balance`. No calendar columns and no `time_idx`.

`dow` is Monday = 0 through Sunday = 6. `month` is 1–12. `hour` is 0–23. A holiday is a Polish public holiday, including 24 December from 2025. A bridge day is a working day with a day off on both sides: the Friday after a Thursday holiday, the Monday before a Tuesday holiday, or the one working day between two holidays. The flag is the same for every hour of that day.

In [6]:
ready = {}
for (model, scope), df in window.items():
    out = df if model == "epf" else add_calendar_features(df)
    if model == "tft":
        out = add_time_idx(out)
    ready[(model, scope)] = out
    print(f"{model}({scope})  {out.shape}  {list(out.columns[-6:])}")

tft(d1)  (37968, 20)  ['hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']
tft(d2)  (37968, 31)  ['hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']
tft(d3)  (37968, 47)  ['hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']
trees(d1)  (37968, 29)  ['eurpln_chg7d', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge']
trees(d2)  (37968, 46)  ['vload_PL_delta24', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge']
trees(d3)  (37968, 62)  ['vload_PL_delta24', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge']
epf(pl)  (37968, 7)  ['PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24', 'PL_solar_fc_lag24', 'PL_gen_fc_lag24', 'PL_gen_fc_balance']


## 5. Train, Validation, Test

`SPLITS` reads the dates from `config.py`. `full` is the modeling window. `train`, `val` and `test` partition it: the next split starts one hour after the previous one ends. All datasets use the same hours. `time_idx` counts from the full window, so validation continues where training ends.

In [7]:
splits = {}
for (model, scope), df in ready.items():
    parts = split_dataset(df)
    splits[(model, scope)] = parts
    sizes = ", ".join(f"{name} {len(part)}" for name, part in parts.items())
    print(f"{model}({scope})  {sizes}  NaN {int(parts['full'].isna().sum().sum())}")

train = splits[("tft", "d1")]["train"]
val = splits[("tft", "d1")]["val"]
print(f"Index range: {train.index.max()} - {val.index.min()}")

tft(d1)  full 37968, train 26208, val 3024, test 8736  NaN 0
tft(d2)  full 37968, train 26208, val 3024, test 8736  NaN 0
tft(d3)  full 37968, train 26208, val 3024, test 8736  NaN 0
trees(d1)  full 37968, train 26208, val 3024, test 8736  NaN 0
trees(d2)  full 37968, train 26208, val 3024, test 8736  NaN 0
trees(d3)  full 37968, train 26208, val 3024, test 8736  NaN 0
epf(pl)  full 37968, train 26208, val 3024, test 8736  NaN 0
Index range: 2025-06-15 23:00:00 - 2025-06-16 00:00:00


## 6. Saving

Twenty-eight files under `data/final/`: TFT and trees for each step as parquet, plus `epf` for Poland as CSV, each as `full`, `train`, `val` and `test`. They cover the same hours. None of the files contain missing values.

`load_final_dataset(model, scope, split)` reads one file.

In [8]:
for model in ("tft", "trees"):
    for step in ("d1", "d2", "d3"):
        columns = splits[(model, step)]["train"].columns
        print(
            f"=== {model} {step.upper()}: {len(columns) - 1} columns after the target ==="
        )
        print(list(columns))

=== tft D1: 19 columns after the target ===
['PL_price', 'PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24', 'PL_solar_fc_lag24', 'PL_gen_fc_lag24', 'PL_coal_gen_lag48', 'ttf_lag48', 'eua_lag48', 'coal_lag48', 'eurpln_lag48', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']
=== tft D2: 30 columns after the target ===
['PL_price', 'PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24', 'PL_solar_fc_lag24', 'PL_gen_fc_lag24', 'PL_coal_gen_lag48', 'ttf_lag48', 'eua_lag48', 'coal_lag48', 'eurpln_lag48', 'np_PL', 'gen_PL', 'vload_PL', 'ac_PL_import', 'ac_PL_export', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL', 'min_PL', 'max_PL', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']
=== tft D3: 46 columns after the target ===
['PL_price', 'PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24

In [9]:
for (model, scope), parts in splits.items():
    for name, part in parts.items():
        path = final_dataset_path(model, scope, name)
        save_merged_data(part, path)
        inspect_merged_file(path)

  [FILE] data/final/tft_d1_full.parquet | shape: (37968, 20) | idx: 2022-06-20 00:00:00-2026-10-18 23:00:00 | cols: 20 | NaN: 0
  [FILE] data/final/tft_d1_train.parquet | shape: (26208, 20) | idx: 2022-06-20 00:00:00-2025-06-15 23:00:00 | cols: 20 | NaN: 0
  [FILE] data/final/tft_d1_val.parquet | shape: (3024, 20) | idx: 2025-06-16 00:00:00-2025-10-19 23:00:00 | cols: 20 | NaN: 0
  [FILE] data/final/tft_d1_test.parquet | shape: (8736, 20) | idx: 2025-10-20 00:00:00-2026-10-18 23:00:00 | cols: 20 | NaN: 0
  [FILE] data/final/tft_d2_full.parquet | shape: (37968, 31) | idx: 2022-06-20 00:00:00-2026-10-18 23:00:00 | cols: 31 | NaN: 0
  [FILE] data/final/tft_d2_train.parquet | shape: (26208, 31) | idx: 2022-06-20 00:00:00-2025-06-15 23:00:00 | cols: 31 | NaN: 0
  [FILE] data/final/tft_d2_val.parquet | shape: (3024, 31) | idx: 2025-06-16 00:00:00-2025-10-19 23:00:00 | cols: 31 | NaN: 0
  [FILE] data/final/tft_d2_test.parquet | shape: (8736, 31) | idx: 2025-10-20 00:00:00-2026-10-18 23:00:00 